In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
from src import encode_meta
from src import narrow_peak
from src import os_func
from src import df_func
from src import mobi
from src import dSQ

### 0. Parameters

In [ ]:
#### User input parameters and paths

## main paras
# name for this run
para_job_id = "GM12878"
# number of top peaks that will be used for all analysis
para_data_n_peaks = 3000
# window size for computing crowdness score
para_crowdness_n_sliding = 250
# number of sequences in the input for motif inference
para_inference_n_seq = 250

## folders
# genome fasta file
para_genome_file = "/home/jg2447/slayman/data/genome/GRCh37.p13.genome.fa"
# dir containing raw ChIP-seq bed from ENCODE
para_chip_data_dir = "/home/jg2447/slayman/data/ENCODE_ChIP_seq/bed/human_hg19/"
# clean ENCODE metadata file
para_metafile = "/home/jg2447/slayman/motif_inference/result/metadata/human-GM12878.txt"
# FIMO result of all known motifs
para_fimo_dir = "/home/jg2447/slayman/data/FIMO/human_stack/FIMOp_0.000100_chr/"
# meme format of all known motifs
para_known_motif_dir = "/home/jg2447/slayman/data/motif/cisbp/meme_human_stack/"

# main result directory
para_result_folder = "/home/jg2447/slayman/motif_inference/result/MOBI/humanGM12878/"

## subfolders under main result directory, could be put outside of main result directory
# all ChIP-seq bed files that will be used
para_TFdata_dir = os.path.join(para_result_folder, "TFdata")
# file containing all summits
para_all_summit_file = os.path.join(para_result_folder, "all_summit.txt")
# containing spp, c-score for each TF
para_crowdness_dir = os.path.join(para_result_folder, "crowdness")
# containing spp, c-score, known-motif-occurance, TF-name for each TF for each desired binding site length
para_motif_count_dir = os.path.join(para_result_folder, "motif_count")
# rank files that aggregate all TF for each ranking, each length
para_rank_file_dir = os.path.join(para_result_folder, "rank_file")
# known motif enrichment directory
para_enrichment_dir = os.path.join(para_result_folder, "enrichment")
# inference input sequences in bed format
para_bed_dir = os.path.join(para_result_folder, "bed_file")
# inference input sequences in fasta format
para_fasta_dir = os.path.join(para_result_folder, "fasta_file")
# inferred motifs
para_motif_dir = os.path.join(para_result_folder, "inference_motif")
# inferred motifs vs known motifs, tomtom result
para_tomtom_dir = os.path.join(para_result_folder, "inference_tomtom")
# tomtom summary
para_tomtom_summary_dir = os.path.join(para_result_folder, "tomtom_summary")
# performance statistics
para_performance_dir = os.path.join(para_result_folder, "performance")

In [ ]:
# all tested ranking types and lengths
alpha_list = np.round(np.append(np.arange(0.1,1.0,0.1), np.arange(1.0,11.0,1.0)), decimals=2)
rank_list = ["RankSPP"]
rank_list.extend(["RankLinear_%.1f" % i for i in alpha_list])
rank_list.extend(["RankCrowdness"])
nbp_list = [10, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200]

### 1. Input ChIP-seq data

In [ ]:
# get TF file names and TF names from metafile
metadata = encode_meta.unique_TF_parser(para_metafile)
u_TF_files, u_TF_names, u_TF_files_w_motif, u_TF_names_w_motif, u_motif_files = metadata[5:]

# collect all ChIPped TF and pick top 3000
# these will be used for all_summit, 
# and all or subset of these 3000 will be used for downstream

TF_files_raw = [os.path.join(para_chip_data_dir,i+".bed.gz") for i in u_TF_files]

In [ ]:
os_func.mkdir_empty(para_TFdata_dir, para_TFdata_dir+" existed")

for i in range(len(TF_files_raw)):
    narrow_peak.get_subset(
        infile=TF_files_raw[i],
        outfile=os.path.join(para_TFdata_dir, u_TF_files[i]+".bed"),
        data_proportion=para_data_n_peaks,
        sort_column=7,
        read_method="zcat")

### 2. Collect summits of all TFBS for crowding score

In [ ]:
# collect all summits from all ChIPped TF (top 3000 peaks of each bed file)
TF_files_for_summit = [os.path.join(para_TFdata_dir, i+".bed") for i in u_TF_files]

if os.path.exists(para_all_summit_file):
    raise OSError("para_all_summit_file: "+para_all_summit_file+" existed")
    
mobi.get_all_summit_file(
    TF_files=TF_files_for_summit, 
    all_summit_file=para_all_summit_file)

### 3. Calculate crowding score for each TFBS

In [ ]:
# Subset of all TF that will be used in the downstream analysis
# here use TFs that have known motifs (so that we could do validation)
TF_files_for_crowdness = [os.path.join(para_TFdata_dir, i+".bed") for i in u_TF_files_w_motif]

# compute crowdness score
os_func.mkdir_empty(para_crowdness_dir, para_crowdness_dir+" existed")

mobi.get_crowdness_sliding_window(
    TF_files=TF_files_for_crowdness,
    result_dir=para_crowdness_dir,
    all_summit_file=para_all_summit_file,
    crowdness_n_sliding=para_crowdness_n_sliding)

### 4. Calculate known motif occurence

In [ ]:
TF_files_w_crowdness = [os.path.join(para_crowdness_dir, i+".bed") for i in u_TF_files_w_motif]
motif_fimo_files = [os.path.join(para_fimo_dir, i+".bed") for i in u_motif_files]

tmp_dir = os_func.mkdir_tmp()

# all test binding site length
for nn in nbp_list:
    sub_result_dir = os.path.join(para_motif_count_dir, str(nn))
    os_func.mkdir_empty(sub_result_dir, sub_result_dir+" existed")

    os.makedirs(os.path.join(tmp_dir, str(nn)))
    
    # trim binding sites to desire length
    for tf, fimo in zip(TF_files_w_crowdness, motif_fimo_files):
        narrow_peak.get_fix_width_region(
            infile=tf,
            outfile=os.path.join(tmp_dir, str(nn), os.path.basename(fimo)),
            width=nn,
            summit_col=None,
            keep_col=[4,5],
            read_method="cat")
    
    # compute known motif occurence
    TF_files_w_crowdness_trimmed = [os.path.join(tmp_dir, str(nn), os.path.basename(i)) for i in motif_fimo_files]
    mobi.get_motif_hit(
        TF_files=TF_files_w_crowdness_trimmed,
        motif_fimo_files=motif_fimo_files,
        result_dir=sub_result_dir)
    
    # remove TFs that fail in all previous processes
    os_func.remove_size_zero_file(sub_result_dir)
shutil.rmtree(tmp_dir)

### 5. Rank TFBS by SPP/C-score/BC-score

In [ ]:
######## rank file ########
os_func.mkdir_empty(para_rank_file_dir, para_crowdness_dir+" existed")

for nn in nbp_list:
    for rank in rank_list:
        # set alpha based on the input Ranking method
        if "_" in rank:
            alpha = float(rank.split("_")[1])
        else:
            alpha = 1
        
        # generate rank file
        TF_files_w_motif_count = [os.path.join(para_motif_count_dir, str(nn), i) for i in os.listdir(os.path.join(para_motif_count_dir, str(nn))) if i.endswith(".bed")]
        mobi.get_rank_file(
            rank_method=rank,
            TF_files=TF_files_w_motif_count,
            result_file="%s/%s_%s.txt" % (para_rank_file_dir, rank, nn),
            bs_half_length=nn,
            rank_linear_alpha=alpha)

### (Optional) Calculate known motif enrichment

In [ ]:
# the ranking+length combinations that will be used
rank_files = [i for i in os.listdir(para_rank_file_dir) if i.endswith(".txt") and not i.startswith(".")]

# calculate enrichment
for rank_file in rank_files:
    for n_seq in range(50,3050,50):
        mobi.get_motif_enrichment(
            rank_file=os.path.join(para_rank_file_dir, rank_file),
            n_seq=n_seq,
            result_dir=os.path.join(para_enrichment_dir, rank_file.replace(".txt", "/")),
            enrichment_method="EnrichmentFractionHalfInsuff")

In [ ]:
# summarize results
mobi.summarize_motif_enrichment(para_enrichment_dir)

### 6. Convert ranked BS bed coordinate to sequence fasta

In [ ]:
for nn in nbp_list:
    for rank in rank_list:
        # extract the input sequences for motif inference from rank files
        mobi.get_bed_from_rank_file(
            rank_file="%s/%s_%s.txt" % (para_rank_file_dir, rank, nn),
            inference_n_seq=para_inference_n_seq,
            inference_nbp=nn,
            result_dir="%s/%s_%d" % (para_bed_dir, rank, nn))
        # convert from bed format to fasta
        mobi.get_fasta_from_bed(
            bed_dir="%s/%s_%d" % (para_bed_dir, rank, nn),
            genome_file=para_genome_file,
            result_dir="%s/%s_%d" % (para_fasta_dir, rank, nn))

# delete TFs with empty fasta
for i in os.listdir(para_fasta_dir):
    if not i.startswith("."):
        os_func.remove_size_zero_file(os.path.join(para_fasta_dir, i))

### 7. Motif inference

In [ ]:
# run inference
# generate joblist

for tool in ["DREME", "HOMER", "MEME"]:
    for nn in nbp_list:
        for rank in rank_list:
            mobi.run_inference_joblist(
                inference_tool=tool,
                fasta_dir="%s/%s_%d/" % (para_fasta_dir, rank, nn),
                hpc_joblist_file="/home/jg2447/log/Inference_%s_%s_%s_%d.txt" % (para_job_id, tool, rank, nn),
                inference_motif_dir="%s/%s_%s_%d/" % (para_motif_dir, tool, rank, nn),
                keep_motif_only=True,
                homerMotif2meme_script="/gpfs/slayman/pi/gerstein/jg2447/motif_inference/MOBI/scripts/R/MoVRs_Motif2meme.R")

In [ ]:
# generate dSQ file
os.system("cat /home/jg2447/log/Inference_%s_*.txt > /home/jg2447/log/%s-inf" % (para_job_id, para_job_id))
os.system("rm /home/jg2447/log/Inference_%s_*.txt" % para_job_id)
dSQ.get_sh("/home/jg2447/log/%s-inf" % para_job_id)

### 8. Compare inferenced motif to known motifs

In [ ]:
# run tomtom
# generate joblist

for tool in ["DREME", "HOMER", "MEME"]:
    for nn in nbp_list:
        for rank in rank_list:
            mobi.run_tomtom_joblist(
                inference_tool=tool,
                fasta_dir="%s/%s_%d/" % (para_fasta_dir, rank, nn),
                hpc_joblist_file="/home/jg2447/log/Tomtom_%s_%s_%s_%d.txt" % (para_job_id, tool, rank, nn),
                inference_motif_dir="%s/%s_%s_%d/" % (para_motif_dir, tool, rank, nn),
                known_motif_dir=para_known_motif_dir,
                inference_tomtom_dir="%s/%s_%s_%d/" % (para_tomtom_dir, tool, rank, nn),
                keep_motif_only=True)

# generate dSQ file
os.system("cat /home/jg2447/log/Tomtom_%s_*.txt > /home/jg2447/log/%s-tom" % (para_job_id, para_job_id))
os.system("rm /home/jg2447/log/Tomtom_%s_*.txt" % para_job_id)
dSQ.get_sh("/home/jg2447/log/%s-tom" % para_job_id)

### 9. Calculate summary statistics from TOMTOM result

In [ ]:
os_func.mkdir_empty(para_tomtom_summary_dir)

for tool in ["DREME", "HOMER", "MEME"]:
    for nn in nbp_list:
        for rank in rank_list:
            prefix = "%s_%s_%d_" % (tool, rank, nn)
            tfs = [os.path.basename(i).replace(".meme", "") for i in os.listdir(para_motif_dir) if i.startswith(prefix) and i.endswith(".meme")]
            meme_file_list = [os.path.join(para_motif_dir, i+".meme") for i in tfs]
            tomtom_file_list = [os.path.join(para_tomtom_dir, i+".tomtom") for i in tfs]
            
            hits = mobi.get_tomtom_result(
                meme_file_list=meme_file_list,
                tomtom_file_list=tomtom_file_list,
                top_nn=5)
            hits.to_csv(
                "%s/%stop5.txt" % (para_tomtom_summary_dir, prefix),
                sep="\t", index=False, header=True, float_format='%.3f')

### 10. Find the optimal parameters (Rank and Length)

In [ ]:
with open(os.path.join(para_result_folder, "optimal_top5.txt"), "w") as f:
    for tool in ["DREME", "HOMER", "MEME"]:
        df = mobi.get_tomtom_summary_data(para_tomtom_summary_dir, nbp_list, rank_list, tool, "top5", improve=False, tf_subset=None)
        idx = df_func.global_max_index(df)
        f.write("%s: %s, %s\n" % (tool, idx[0], idx[1]))

### (Optional) Calculate performance evaluation statistics

In [ ]:
para_tomtom_summary_dir = '/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanGM12878/tomtom_summary'

In [ ]:
from sklearn.model_selection import KFold

# list of TFs
TFs = pd.read_csv(os.path.join(para_tomtom_summary_dir, "DREME_RankSPP_100_count_top5.txt"), 
                  sep="\t")["TF"].values
# collect number of known motifs for each TF
n_known_TFs = []
for tf in TFs:
    with open("%s/%s.meme" % (para_known_motif_dir, tf), "r") as f:
        mf = f.readlines()
    n_known = len([i for i in mf if i.startswith("MOTIF")])
    n_known_TFs.append(n_known)
n_known_TFs = np.array(n_known_TFs)

In [ ]:
DREME_result = []
HOMER_result = []
MEME_result = []

# subsample and calculate the performance statistics for 10k times
for rs in range(2000):
    kf = KFold(n_splits=5, shuffle=True, random_state=rs)
    for i,j in kf.split(TFs):
        for tool in ["DREME", "HOMER", "MEME"]:
            
            # use subset i to get optimal parameters
            df = mobi.get_tomtom_summary_data(para_tomtom_summary_dir, nbp_list, rank_list, tool, "top5", improve=False, tf_subset=TFs[i])
            idx = df.global_max_index(df)
            
            # calculate the results in subset j as testing performance
            optimal_count = pd.read_csv("%s/%s_%s_%s_count_top5.txt" % (para_tomtom_summary_dir, tool, idx[0], idx[1]), sep="\t")
            optimal_count = optimal_count[optimal_count["TF"].isin(TFs[j])].sort_values("TF").reset_index(drop=True).copy()

            spp_count = pd.read_csv("%s/%s_RankSPP_100_count_top5.txt" % (para_tomtom_summary_dir, tool), sep="\t")
            spp_count = spp_count[spp_count["TF"].isin(TFs[j])].sort_values("TF").reset_index(drop=True).copy()

            diff = (optimal_count[1] - spp_count[1]).values

            optimal_known_hit = pd.read_csv("%s/%s_%s_%s_known_hit_top5.txt" % (para_tomtom_summary_dir, tool, idx[0], idx[1]), sep="\t")
            optimal_known_hit = optimal_known_hit[optimal_known_hit["TF"].isin(TFs[j])].sort_values("TF").reset_index(drop=True).copy()

            spp_known_hit = pd.read_csv("%s/%s_RankSPP_100_known_hit_top5.txt" % (para_tomtom_summary_dir, tool), sep="\t")
            spp_known_hit = spp_known_hit[spp_known_hit["TF"].isin(TFs[j])].sort_values("TF").reset_index(drop=True).copy()

            diff_known = (optimal_known_hit[1] - spp_known_hit[1]).values

            result = [spp_count[1].mean(), # spp correctly inf motif
                      optimal_count[1].mean(), # optimal correctly inf motif
                      np.sum(diff > 0), # fraction of increase inf motif
                      np.sum(diff == 0), # fraction of noChange inf motif
                      np.sum(diff < 0), # fraction of decrease inf motif
                      spp_known_hit[1].mean(), # spp known motif hit
                      optimal_known_hit[1].mean(), # optimal known motif hit
                      np.sum(diff_known > 0), # fraction of increase known motif hit
                      np.sum(diff_known == 0), # fraction of noChange known motif hit
                      np.sum(diff_known < 0), # fraction of decrease known motif hit
                      (spp_known_hit[1]/n_known_TFs[j]).mean(), # spp known motif coverage percentage
                      (optimal_known_hit[1]/n_known_TFs[j]).mean()] # optimal known motif coverage percentage
            
            if tool == "DREME":
                DREME_result.append(result)
            elif tool == "HOMER":
                HOMER_result.append(result)
            elif tool == "MEME":
                MEME_result.append(result)

# save result to file
DREME_result = pd.DataFrame(DREME_result)
DREME_result.to_csv("%s/DREME_top5.txt" % (para_performance_dir), sep="\t", header=False, index=False, float_format="%.3f")
HOMER_result = pd.DataFrame(HOMER_result)
HOMER_result.to_csv("%s/HOMER_top5.txt" % (para_performance_dir), sep="\t", header=False, index=False, float_format="%.3f")
MEME_result = pd.DataFrame(MEME_result)
MEME_result.to_csv("%s/MEME_top5.txt" % (para_performance_dir), sep="\t", header=False, index=False, float_format="%.3f")